## Setup

In [ ]:
import pandas as pd  # tables of data

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## 2026: Poverty rate by race and Hispanic origin and NEET status

In [ ]:
race_rows = {"White, non-Hispanic": "white", "Black, non-Hispanic": "black", "Asian, non-Hispanic": "asian", "Hispanic (any race)": "hispanic", "Other, non-Hispanic": "other_race"}  # row labels and their race dummies, in order

def wmean(data, var):  # weighted mean of one variable
    d = data.dropna(subset=[var])  # leave out people with a missing value
    return (d[var] * d["asecwt"]).sum() / d["asecwt"].sum()  # sum of value x weight, divided by sum of weights

year26 = df[df["year"] == 2026]  # keep only the 2026 survey

race_count = year26[list(race_rows.values())].sum(axis=1)  # number of race dummies equal to 1, for each person
n_not_one = (race_count != 1).sum()  # people without exactly one race dummy equal to 1
print("Check: every person in 2026 has exactly one race dummy equal to 1:", bool(n_not_one == 0))  # True or False
print("People without exactly one race dummy equal to 1:", n_not_one)  # should be 0

label_for = df.dropna(subset=["neet", "neet_label"]).drop_duplicates("neet").set_index("neet")["neet_label"]  # neet_label for each value of neet
groups = [(1, label_for[1]), (0, label_for[0])]  # (neet value, column name): NEET first, then Not NEET

known = year26.dropna(subset=["poor", "neet"])  # leave out people with poor or neet missing
print("People in 2026 left out because poor or neet is missing (unweighted):", len(year26) - len(known))  # how many were left out
print()  # empty line

rate = {}  # weighted poverty rate, one row per race group
people = {}  # unweighted respondents, same layout
for race_label, race_var in race_rows.items():  # loop over the five race groups
    rate[race_label] = {}  # start this row of the table
    people[race_label] = {}  # start this row of the counts
    for neet_value, column_name in groups:  # loop over NEET and Not NEET
        cell = known[(known[race_var] == 1) & (known["neet"] == neet_value)]  # people in this cell
        rate[race_label][column_name] = wmean(cell, "poor") * 100  # weighted percent poor
        people[race_label][column_name] = len(cell)  # unweighted respondents in this cell

table = pd.DataFrame(rate).T  # dictionary -> table, one row per race group
people = pd.DataFrame(people).T  # same layout for the counts
table["Gap (percentage points)"] = table[groups[0][1]] - table[groups[1][1]]  # NEET rate minus Not NEET rate, before rounding

shown = table.copy().astype(object)  # text version of the table, for the asterisks
for row in table.index:  # loop over the rows
    small = []  # which of the two cells in this row have fewer than 30 respondents
    for column_name in people.columns:  # loop over the NEET and Not NEET columns
        is_small = people.loc[row, column_name] < 30  # fewer than 30 unweighted respondents
        small.append(is_small)  # remember it for the gap
        shown.loc[row, column_name] = f"{table.loc[row, column_name]:.1f}" + ("*" if is_small else " ")  # percent, then an asterisk or a space
    shown.loc[row, "Gap (percentage points)"] = f"{table.loc[row, 'Gap (percentage points)']:.1f}" + ("*" if any(small) else " ")  # gap, asterisk if either cell is small

print("Table. Poverty rate (SPM), % poor, by race and Hispanic origin and NEET status, 2026 (weighted with asecwt). * = fewer than 30 unweighted respondents.")  # table title
print(shown.to_string())  # table, 1 decimal, asterisks
print()  # empty line
print("Unweighted number of respondents in each cell.")  # counts title
print(people.to_string())  # counts, same layout
print()  # empty line
print("Poverty status is based on 2025 income; NEET status refers to the March 2026 survey week. Poverty is measured for the whole household, not the individual.")  # required note

**What one cell means.** Take the row "Hispanic (any race)" and the column "NEET". Among Hispanic people in the 2026 survey who were NEET (not employed and not in school) and whose poverty status is known, the cell is the percent who were poor, by the Supplemental Poverty Measure. The column "Not NEET" is the same percent for Hispanic people who were not NEET. The last column, "Gap (percentage points)", is the NEET percent minus the Not NEET percent in that row; a positive gap means poverty is more common among NEET people. People are counted by how many people in the country they represent (`asecwt`). Poverty is measured for the whole household, not the individual. An asterisk marks a cell with fewer than 30 survey respondents behind it, and a gap gets an asterisk if either of its two cells does. The count table under the table shows the number of respondents in each cell.